# 第 6 章 · SwiGLU 前馈网络

> 📄 配套动画网页：`../06_swiglu.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第二部分 · 模型

注意力负责 token 之间的交流，前馈网络负责单个 token 内部的加工。SwiGLU 用一个「门」决定哪些信息通过。

---


## 🎯 本章目标

- 理解 GLU（门控线性单元）的结构
- 掌握 `nn.Linear(bias=False)`、`nn.SiLU`、逐元素乘
- 会从 `transformers` 的 `ACT2FN` 查激活函数
- 能从零实现 `FeedForward`（SwiGLU）

---


## 🖼 官方配图

![前馈网络在整体架构中的位置：每个 Block 里注意力之后就是它](../images/LLM-structure.jpg)

*前馈网络在整体架构中的位置：每个 Block 里注意力之后就是它*

---


## 📖 背景：门控前馈

传统 FFN 是 `d → 4d → d`（两个矩阵）。SwiGLU 变成三个矩阵，中间多了一条「门控」分支：

```python
gate = SiLU(W_gate @ x)     # 门：决定放行多少
up   =        W_up   @ x     # 内容：决定传什么
h    = gate * up             # 逐元素相乘 = 门控
out  = W_down @ h            # 降回 hidden_size
```

因为多了一个矩阵，中间维度要缩小到约 `8/3·d` 才能保持参数量相当。
MiniMind 用 `ceil(d·π/64)·64`，768 维时得到 2432（对齐 64 的倍数，矩阵乘更快）。

---



# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 6.1 `nn.Linear` 与 `bias=False`

现代 LLM 的线性层几乎都不带偏置——省参数，且归一化已经处理了平移。

In [ ]:
import torch
from torch import nn

lin = nn.Linear(4, 8, bias=False)
print("weight 形状:", lin.weight.shape)      # [out, in]
print("有 bias 吗  :", lin.bias)

x = torch.randn(2, 4)
print("输出形状    :", lin(x).shape)

### 6.2 SiLU 与 `nn.SiLU`

`SiLU(x) = x · sigmoid(x)`，比 ReLU 平滑，负半轴不会硬性归零。

In [ ]:
import torch.nn.functional as F

x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
silu_manual = x * torch.sigmoid(x)
silu_nn = nn.SiLU()(x)

print("手动 SiLU:", silu_manual.tolist())
print("nn.SiLU  :", silu_nn.tolist())
print("一致吗    :", torch.allclose(silu_manual, silu_nn))
print("对比 ReLU :", F.relu(x).tolist(), " <- 负半轴直接变 0")

### 6.3 用 `ACT2FN` 查表

`transformers` 提供了「激活函数名 → 函数」的映射表，方便从配置字符串拿到激活函数。

In [ ]:
try:
    from transformers.activations import ACT2FN
    print("可用激活函数:", list(ACT2FN.keys()))
    print("silu ->", ACT2FN["silu"])
except ImportError:
    print("未安装 transformers，可用 uv add transformers 安装")

### 6.4 逐元素乘：门控的本质

`gate * up` 是逐元素相乘，不是矩阵乘。这是「门」能选择性放行的关键。

In [ ]:
gate = torch.tensor([[0.9, 0.1, 0.05]])
up   = torch.tensor([[2.0, 2.0, 2.0]])
print("gate   :", gate.tolist())
print("up     :", up.tolist())
print("门控结果:", (gate * up).tolist(), " <- 门值小的地方被压下去了")

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

模型部分只给提示。请自己写出 SwiGLU 前馈层。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 6.5 实现 `FeedForward`

**要求**：
- `__init__(self, config, intermediate_size=None)`：创建 `gate_proj`、`up_proj`（`hidden → intermediate`）
  和 `down_proj`（`intermediate → hidden`），都不带 bias
- 从 `config.hidden_act` 用 `ACT2FN` 取激活函数
- `forward(x)`：返回 `down_proj(act(gate_proj(x)) * up_proj(x))`

**小坑**：`gate_proj` 和 `up_proj` 的输出维度必须一致才能相乘。

In [ ]:
# ======== 你的实现：FeedForward ========
#
# 【要求】
#   - 继承 nn.Module
#   - intermediate_size 默认取 config.intermediate_size
#   - 三个 nn.Linear(..., bias=False)：gate_proj / up_proj / down_proj
#   - self.act_fn = ACT2FN[config.hidden_act]
#   - forward: down_proj(act_fn(gate_proj(x)) * up_proj(x))
#
# 【可用到的函数 / 方法】
#   - `nn.Linear(in_features, out_features, bias=False)`
#   - `ACT2FN`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：FeedForward ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)

    class Cfg:
        hidden_size = 16
        intermediate_size = 32
        hidden_act = "silu"

    ffn = FeedForward(Cfg())
    x = torch.randn(2, 7, 16)
    y = ffn(x)
    assert tuple(y.shape) == (2, 7, 16), f"输出形状应为 (2, 7, 16)，得到 {tuple(y.shape)}"
    n = sum(p.numel() for p in ffn.parameters())
    assert n == 3 * 16 * 32, f"参数量应为 3×16×32 = {3 * 16 * 32}，得到 {n}"
    ref = ffn.down_proj(F.silu(ffn.gate_proj(x)) * ffn.up_proj(x))
    close(y, ref, tol=1e-5, msg="与 down_proj(silu(gate_proj(x)) * up_proj(x)) 不一致")

check("FeedForward 形状 / 参数量 / 与公式一致", _t)

### 6.6 验证形状与参数量

写完之后：
1. 输入 `[B, S, hidden]`，输出形状应该不变；
2. 打印参数量，和 `2 × d × intermediate + d × intermediate` 对一下。

In [ ]:
# ======== 你的实现：verify_ffn ========
#
# 【要求】
#   - 构造一个 config 对象（含 hidden_size / intermediate_size / hidden_act）
#   - 实例化你的 FeedForward
#   - 输入 torch.randn(2, 10, hidden_size)，确认输出形状相同
#   - 统计参数量并与理论值比较
#
# 【可用到的函数 / 方法】
#   - `sum(p.numel() for p in ffn.parameters())`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能解释 gate / up / down 三个矩阵各自的作用
- [ ] 知道为什么中间维度是 8/3·d 而不是 4d
- [ ] 能说清 SiLU 比 ReLU 好在哪里
- [ ] 参数量与理论值对上了吗？

**思考题**：如果把 `gate_proj` 和 `up_proj` 合并成一个矩阵（一次投影到 2×intermediate 再切开），计算效率会变吗？
